# Cross-validation — how robust is the honest R²?

A single train/test split gives ONE number; it could be lucky or unlucky. **Leakage-safe 5-fold
cross-validation** trains the model on 5 different **station-disjoint** folds (GroupKFold on
`station_id`, so a station is never in its own fold's training data) and reports R²/MAE/coverage as a
**mean ± 95% CI**. This turns "R²=0.22" into "R²=0.22 ± CI" — the robustness check a reviewer expects.

> ⏳ This trains 5 models in one go (~5–6 h on a T4). Run it as **Save & Run All (Commit)** and walk
> away. To go faster, set `N_SPLITS = 3` below (~3 h).

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Kaggle bootstrap — RUN ME FIRST ===
# In the right sidebar: Accelerator = GPU T4, Internet = ON (needs a phone-verified account).
REPO_URL = "https://github.com/keyaan01/pm25-visual-aq.git"   # <-- your repo

import os, sys, shutil, subprocess
BASE = "/kaggle/working"
REPO = os.path.join(BASE, "pm25-visual-aq")
os.chdir(BASE)
shutil.rmtree(REPO, ignore_errors=True)          # ALWAYS start clean -> never a stale/rogue clone
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
assert os.path.exists(os.path.join(REPO, "src", "ceiling.py")), "clone incomplete (is Internet On?) -- src/ceiling.py missing"
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)

import torch
_head = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo:", os.getcwd(), "| HEAD:", _head)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator=GPU")

In [ ]:
import os, json, numpy as np, pandas as pd
from src.config import load_config
from src import data, physics, crossval
cfg = load_config()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
WORK = "/kaggle/working"
cache_path = os.path.join(WORK, "pm25_cache", "physics_maps_%d.npy" % cfg["data"]["image_size"])
out_root = os.path.join(WORK, "pm25_outputs"); os.makedirs(os.path.dirname(cache_path), exist_ok=True)

ds, df = data.load_clean(cfg["data"]["hf_repo"], from_disk=False, seed=cfg["seed"])
print("rows:", len(df))
if not os.path.exists(cache_path):
    physics.build_map_cache(len(ds), lambda i: data.get_image(ds, i), cache_path,
        size=cfg["data"]["image_size"], patch=cfg["physics"]["dcp_patch"],
        omega=cfg["physics"]["dcp_omega"], top_frac=cfg["physics"]["atmos_top_frac"], t_min=cfg["physics"]["t_min"])
cache = physics.load_map_cache(cache_path)

## Run leakage-safe grouped cross-validation

Same model + config as the honest single-split run — only the fold assignment changes. Every fold is
station-disjoint (its `straddling` column must be 0).

In [ ]:
N_SPLITS = 5   # set to 3 for a ~3 h run
reports, summary = crossval.run_cv(ds, df, cache, cfg, device, out_root, n_splits=N_SPLITS, verbose=False)

cols = ["fold","R2","r2_pearson","MAE","RMSE","Spearman","coverage","SD_y_test","n_train","n_test","straddling_stations"]
per_fold = pd.DataFrame(reports)[cols].round(3)
per_fold.to_csv(os.path.join(out_root, "cv_folds.csv"), index=False)
json.dump(summary, open(os.path.join(out_root, "cv_summary.json"), "w"), indent=2, default=float)
per_fold

## The headline: mean ± 95% CI across folds

In [ ]:
for k in ["R2","MAE","RMSE","Spearman","coverage"]:
    s = summary[k]
    print("%-9s mean=%.3f  std=%.3f  95%% CI [%.3f, %.3f]  (n=%d folds)"
          % (k, s["mean"], s["std"], s["ci_lo"], s["ci_hi"], s["n"]))
print()
sr = summary["R2"]
print("Report as: honest R2 = %.3f +/- %.3f  (95%% CI [%.3f, %.3f]) across %d station-disjoint folds"
      % (sr["mean"], sr["mean"]-sr["ci_lo"], sr["ci_lo"], sr["ci_hi"], sr["n"]))

## Done — paste me the per-fold table + the mean ± CI line

This turns the single-split 0.22 into a robust interval across station-disjoint folds. Save Version
to keep `cv_folds.csv` + `cv_summary.json`.